# B2-024 — Practice p21

*65 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261022`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C3-gradient-descent`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C10-competition-craft`, `book1:C11-neural-training`, `book1:C12-classical-models`, `B2-023-generative-models-diffusion`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C3-gradient-descent](../../../../book1/units/C3-gradient-descent/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C10-competition-craft](../../../../book1/units/C10-competition-craft/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [book1:C12-classical-models](../../../../book1/units/C12-classical-models/lesson.ipynb), [B2-023-generative-models-diffusion](../../B2-023-generative-models-diffusion/lesson.ipynb).

**Data boundary:** use literal unit data and the seeded unit generator (`scripts/generate_capstone_data.py`, read through `data/capstone_data.py`), with the standard library, NumPy, and Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints you did not create in the same run, or student data.

**Set:** C · **Type:** integrative · **Difficulty:** core · **Time budget:** 65 minutes  
**Concepts:** open-ended-model-evaluation

## Task

Produce an evaluation report comparing two model families on the inverse task, with every comparison on validation and one final test call.

**Families.** Both take raw magnitudes, use `log` features standardized with training statistics (computed inside `fit`), and predict `(x, y, strength)`:

* `shallow`: `nn.Sequential(nn.Linear(8, 64), nn.ReLU(), nn.Linear(64, 3))`;
* `deep`: `nn.Sequential(nn.Linear(8, 32), nn.ReLU(), nn.Linear(32, 32), nn.ReLU(), nn.Linear(32, 32), nn.ReLU(), nn.Linear(32, 3))`.

Write `class InverseRegressor` with constructor `InverseRegressor(family, use_log, seed, steps, budget)` and methods `fit(train_x, train_y)` and `predict(x)`. `fit` computes the feature statistics from `train_x`, calls `torch.manual_seed(seed)`, builds the network, wraps `torch.optim.Adam(net.parameters(), lr=3e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0)` with `budget.wrap`, and runs `steps` full-batch MSE updates; `predict` returns `(N, 3)` under `torch.no_grad()`. With `use_log=False` the raw magnitudes are standardized instead.
**Seam:** `fit` records `capstone_data.split_of("inverse", row)` for every row of `train_x` on entry; every recorded value must be `"train"`.

**Budget.** One `budget = capstone_data.StepBudget(2000)` for the whole report. Runs: both families with `use_log=True`, seeds `(0, 1, 2)`, 250 steps each (1,500 steps); the ablation `deep` with `use_log=False`, seeds `(0, 1)`, 250 steps (500 steps).

**Report (validation only).** Implement `bootstrap_ci` and `paired_bootstrap_diff` (p08 contract; metric = mean position error, `n_boot = 1000`, `alpha = 0.05`, each with a fresh `torch.Generator().manual_seed(SEED)`).

1. *Per-family table:* mean and sample standard deviation over seeds of the validation error, and parameter counts.
2. *Intervals:* `bootstrap_ci` for each family's seed-0 model.
3. *Paired comparison across shared seeds:* the three per-seed differences `deep - shallow` of the mean validation errors, and `paired_bootstrap_diff(lambda target, err: err.mean(), torch.zeros(200), err_deep, err_shallow, ...)`, where `err_deep` and `err_shallow` are each family's per-example validation errors averaged over its three seeds (the first argument is unused by this metric).
4. *Ablation table:* `deep` with and without log features over seeds `(0, 1)`.
5. *Robustness:* multiply the validation magnitudes by `1 + 0.05 * eps`, with `eps = torch.randn(200, 8, generator=torch.Generator().manual_seed(SEED + 1))`, and report each family's seed-0 error on the shifted inputs.

**Selection rule (pre-registered).** Choose the family with the lower mean validation error over seeds; if the paired interval contains 0, choose `shallow` (fewer parameters).

**Evaluation protocol.** Fit only on rows from `capstone_data.train_rows("inverse")` (and any other rows this statement names), make every choice on `capstone_data.val_rows("inverse")`, and reach the locked test rows only through `capstone_data.final_test_score(predict_fn, n_boot=1000, alpha=0.05, task="inverse")`, called exactly once, after all fitting and selection. Assert `capstone_data.test_call_count("inverse") == 0` immediately before that call and `== 1` after it.
Score the chosen family's seed-0 model.

**Write** the evaluation section (protocol, headline with interval and the baseline `0.2137`, evidence, robustness, limitations) in at most twelve sentences.

**Certify.** `budget.used == 2000`; every seam value is `"train"`; in each interval `low <= point <= high`; the chosen family follows the rule applied to your numbers; the test score `< capstone_data.baseline_score("inverse").score`; the test-call protocol.

**Required answer-check assertions.** Every certificate item and the shapes of the tables (2 families × 3 seeds; ablation 2 × 2).

In [ ]:
import importlib.util
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20261022
LOADER_CANDIDATES = (
    Path("../data/capstone_data.py"),
    Path("units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
    Path("book2/units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
)
loader_path = next(path for path in LOADER_CANDIDATES if path.is_file())
loader_spec = importlib.util.spec_from_file_location("capstone_data", loader_path)
capstone_data = importlib.util.module_from_spec(loader_spec)
loader_spec.loader.exec_module(capstone_data)
torch.set_num_threads(1)

FAMILIES = ("shallow", "deep")
SEEDS = (0, 1, 2)
ABLATION_SEEDS = (0, 1)
STEPS_PER_RUN = 250
TOTAL_BUDGET = 2000

In [ ]:
# Write the required implementation, probes, certificates, and answer-check assertions here.

## Your response

Show every requested derivation, implementation, probe, certificate, or audit. Use only the permitted data, keep the training, validation, and locked-test roles separate, and end coding work with an answer-check section of executable assertions.